In [1]:
from typing import Annotated
from langchain_groq import ChatGroq
from langchain_core.messages import AnyMessage,AIMessage,BaseMessage
from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt,Command
from dotenv import load_dotenv
from typing import TypedDict,Annotated


d:\Langgraph\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

True

In [3]:
llm=ChatGroq(model="llama-3.3-70b-versatile")

In [4]:
from langgraph.graph.message import add_messages

class ChatState[TypedDict]:
    messages:Annotated[list[BaseMessage],add_messages]

In [5]:
def chat_node(state:ChatState):

    decision=interrupt({
        'type':'approval',
        'reason':'Model is about to answer user question',
        'question':state['messages'][-1].content,
        'instruction':'approve this question? yes/no'
    })

    if decision['approved']=='no':
        return{'messages' :[AIMessage(content="not approved")]}

    else:
        response = llm.invoke(state['messages'])
        return{'messages':[response]}

In [ ]:
# build the graph

builder=StateGraph[ChatState]

builder.add_node('chat',chat_node)
